# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The dataset contains structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility. We will walk through loading, overview, extraction, processing, and visualization of the data using Croissant schema IDs.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. For this, we use the schema URL to initialize a Dataset object and display the dataset's name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print name and description using Croissant metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields. This overview helps us to select which record sets and fields to extract for data analysis.

Let's enumerate the record sets present in the dataset and list their field and column IDs.

In [ ]:
# List all RecordSets and their fields by @id
record_sets = dataset.metadata.record_set  # list of mlc.RecordSet objects
print("Record Sets and their fields (all referenced by @id):")

overview = {}
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    overview[rs.id] = []
    for field in rs.field:
        field_id = field.id
        print(f"    Field @id: {field_id}, Name: {field.name}, DataType: {getattr(field, 'data_type', None)}")
        overview[rs.id].append(field_id)
    if hasattr(rs, 'column') and rs.column:
        for col in rs.column:
            print(f"    Column @id: {col.id}, Name: {col.name}, DataType: {getattr(col, 'data_type', None)}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, using `@id` fields. Use the overview above to select record sets and fields.

This step collects records from all record sets (tables) available in the dataset, referencing each by its `@id`.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

# Load all rows for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))  # Each record is a dict with field @ids
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display info for each record set
for record_set_id in record_set_ids:
    print(f"\nRecordSet @id: {record_set_id}")
    print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All operations reference relevant fields by their `@id`.

For demonstration, we'll:
- Pick a record set (table) with patient clinical features.
- Filter patients by age at diagnosis (`@id` for "age at diagnosis").
- Normalize the age feature.
- Group by presence of a phenotype (e.g., hirsutism, using its `@id`).

In [ ]:
# Choose a record set with clinical features (update these ids to match your dataset schema)
clinical_rs_id = None
for rs in record_sets:
    if 'diagnosis' in rs.name.lower() or 'clinical' in rs.name.lower():
        clinical_rs_id = rs.id
        break
if not clinical_rs_id:
    clinical_rs_id = record_set_ids[0]  # fallback: first record set

clinical_df = dataframes[clinical_rs_id]

# Identify the @id for "age at diagnosis" field
age_field_id = None
hirsutism_field_id = None
for rs in record_sets:
    if rs.id == clinical_rs_id:
        for field in rs.field:
            if "age" in field.name.lower():
                age_field_id = field.id
            if "hirsutism" in field.name.lower():
                hirsutism_field_id = field.id
        break

# Filter records by age (e.g., age at diagnosis > 25)
if age_field_id:
    threshold = 25
    filtered_df = clinical_df[clinical_df[age_field_id] > threshold]
    print(f"Filtered records with {age_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the age feature
    filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
    print(f"Normalized {age_field_id} for filtered records:")
    display(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

    # Group by hirsutism field
    if hirsutism_field_id and hirsutism_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(hirsutism_field_id)[age_field_id].mean().reset_index()
        print(f"Grouped data by {hirsutism_field_id} (mean age):")
        display(grouped_df)
else:
    print("Unable to find field @id for age at diagnosis. Please check dataset schema.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Reference all visualization fields using their `@id`.

We will plot the normalized age distribution, grouped by hirsutism presence, using matplotlib.

In [ ]:
import matplotlib.pyplot as plt

if age_field_id and hirsutism_field_id and age_field_id+"_normalized" in filtered_df.columns:
    plt.figure(figsize=(6, 4))
    for group_val in filtered_df[hirsutism_field_id].unique():
        group = filtered_df[filtered_df[hirsutism_field_id] == group_val]
        plt.hist(group[f"{age_field_id}_normalized"], label=f"hirsutism={group_val}", alpha=0.5)
    plt.xlabel(f"Normalized {age_field_id}")
    plt.ylabel("Count")
    plt.title(f"Age at diagnosis distribution grouped by hirsutism ({hirsutism_field_id})")
    plt.legend()
    plt.show()
else:
    print("Visualization fields not available. Please verify dataset fields.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id`. We:
- Loaded metadata and records
- Provided an overview of available record sets and fields
- Extracted and processed data using field `@id`s
- Performed basic EDA: filtering, normalization, grouping
- Visualized feature distributions

The approach demonstrated how to interact with Croissant datasets, ensuring transparent, reproducible workflows and compatibility for advanced FAIR data analysis.